In [1]:
#%%
import torch
import onnxruntime as ort
import os
import numpy as np
import onnx
import time
import itertools

# ===============================
# CONFIG
# ===============================
lat_perturb = False  # ✅ 위도별 표준편차(=axis=3 기반 perturbation) 활성화 옵션
pangu_dir = r'/home1/jek/Pangu-Weather'

lat_indices = np.linspace(90, -90, 721)
lon_indices = np.linspace(-180, 180, 1441)[:-1]

def latlon_extent(lon_min, lon_max, lat_min, lat_max):    
    lon_min, lon_max = lon_min-180, lon_max-180  
    lat_start = np.argmin(np.abs(lat_indices - lat_max)) 
    lat_end = np.argmin(np.abs(lat_indices - lat_min))
    lon_start = np.argmin(np.abs(lon_indices - lon_min))
    lon_end = np.argmin(np.abs(lon_indices - lon_max))
    latlon_ratio = (lon_max-lon_min)/(lat_max-lat_min)
    extent=[lon_min, lon_max, lat_min, lat_max]
    return lat_start, lat_end, lon_start, lon_end, extent, latlon_ratio

lat_start, lat_end, lon_start, lon_end, extent, latlon_ratio = latlon_extent(100,160,5,45)  # ✅ 저장할 위경도 범위

year = ['2022']
month = ['08']
day = ['27']
times = ['00']
ens_list = range(10,1333)
perturbation_scale_list =[0.05]
factor_list_list = [['z']]

surface_factor = ['MSLP', 'U10', 'V10', 'T2M']
surface_dict = {'MSLP':0, 'U10':1, 'V10':2, 'T2M':3}
upper_factor = ['z', 'q', 't', 'u', 'v']
upper_dict = {'z':0, 'q':1, 't':2, 'u':3, 'v':4}


# ===============================
# ONNX Sessions
# ===============================
options = ort.SessionOptions()
options.enable_cpu_mem_arena= True
options.enable_mem_pattern = False
options.enable_mem_reuse = False

cuda_provider_options_gpu0 = {'arena_extend_strategy': 'kSameAsRequested', 'device_id': 0}
ort_session_6 = ort.InferenceSession(
    rf'{pangu_dir}/pangu_weather_6.onnx', 
    sess_options=options, 
    providers=[('CUDAExecutionProvider', cuda_provider_options_gpu0)]
)

# ===============================
# MAIN LOOP
# ===============================
start = time.time()

for factor_list in factor_list_list:
    for perturbation_scale in perturbation_scale_list:
        for y, m, d, tm in itertools.product(year, month, day, times):
            time_str = f'{y}/{m}/{d}/{tm}UTC'

            input_data_dir = rf'{pangu_dir}/input_data/{time_str}'
            output_data_dir = rf'/data09/Pangu_TC_ENS/output_data/{time_str}'

            input_upper = np.load(os.path.join(input_data_dir, 'upper.npy')).astype(np.float32)
            input_surface = np.load(os.path.join(input_data_dir, 'surface.npy')).astype(np.float32)

            # ===============================
            # STD DEV 계산
            # ===============================
            # 전체 영역 기반 표준편차
            std_dev_upper = np.std(input_upper, axis=(2, 3), dtype=np.float32) * perturbation_scale  # (C, L)
            std_dev_surface = np.std(input_surface, axis=(1, 2), dtype=np.float32) * perturbation_scale

            factor_str = "".join([f"_{f}" for f in factor_list])
            lp_suffix = "_lp" if lat_perturb else ""
            ens_root_dir = rf'/data09/Pangu_TC_ENS/output_data/{time_str}/{perturbation_scale}ENS{factor_str}_6h{lp_suffix}'

            # ===============================
            # ENSEMBLE LOOP
            # ===============================
            for ens in ens_list:
                output_data_dir = os.path.join(ens_root_dir, str(ens))
                os.makedirs(os.path.join(output_data_dir, 'upper'), exist_ok=True)
                os.makedirs(os.path.join(output_data_dir, 'surface'), exist_ok=True)
                
                perturbed_upper = input_upper.copy()
                perturbed_surface = input_surface.copy()

                seed_val = hash((ens, tuple(factor_list), perturbation_scale)) % (2**32)
                rng = np.random.default_rng(seed_val)

                # ================
                # Perturbation
                # ================
                if ens != 0:
                    for factor in factor_list:
                        if factor in upper_dict:
                            idx = upper_dict[factor]
                            for j in range(13):
                                if lat_perturb:
                                    # 위도별 표준편차 적용
                                    noise = rng.normal(0, 1, size=input_upper[idx, j].shape).astype(np.float32)
                                    scale = std_dev_upper[idx, j][..., None]  # (lat, 1)
                                    perturbation = noise * scale
                                else:
                                    # 전체 평균 표준편차 적용
                                    perturbation = rng.normal(0, std_dev_upper[idx, j], input_upper[idx, j].shape)
                                perturbed_upper[idx, j] = input_upper[idx, j] + perturbation.astype(np.float32)

                        elif factor in surface_dict:
                            idx = surface_dict[factor]
                            perturbation = rng.normal(0, std_dev_surface[idx], input_surface[idx].shape)
                            perturbed_surface[idx] = input_surface[idx] + perturbation.astype(np.float32)

                # ================
                # Save initial
                # ================
                np.save(os.path.join(output_data_dir, f'upper/0h'), perturbed_upper[:,:,lat_start: lat_end+1, lon_start:lon_end+1])
                np.save(os.path.join(output_data_dir, f'surface/0h'), perturbed_surface[:,lat_start: lat_end+1, lon_start:lon_end+1])

                start = time.time()

                # ================
                # Forecast loop
                # ================
                for i in range(28):
                    start_i = time.time()
                    predict_interval = 6 * (i + 1)

                    output, output_surface = ort_session_6.run(None, {'input': perturbed_upper, 'input_surface': perturbed_surface})

                    np.save(os.path.join(output_data_dir, f'upper/{predict_interval}h'), output[:,:,lat_start: lat_end+1, lon_start:lon_end+1])
                    np.save(os.path.join(output_data_dir, f'surface/{predict_interval}h'), output_surface[:,lat_start: lat_end+1, lon_start:lon_end+1])

                    perturbed_upper, perturbed_surface = output, output_surface
                    end_i = time.time()
                    print(f'{factor_list} {perturbation_scale}_{ens}ENS {i+1}번째 반복 +{predict_interval}h {end_i-start_i:.2f}s')

                end = time.time()
                print(f"{factor_list} {perturbation_scale}_{ens}ENS: {end-start:.1f}s")


['z'] 0.05_10ENS 1번째 반복 +6h 1.08s
['z'] 0.05_10ENS 2번째 반복 +12h 0.96s
['z'] 0.05_10ENS 3번째 반복 +18h 0.96s
['z'] 0.05_10ENS 4번째 반복 +24h 0.96s
['z'] 0.05_10ENS 5번째 반복 +30h 0.96s
['z'] 0.05_10ENS 6번째 반복 +36h 0.96s
['z'] 0.05_10ENS 7번째 반복 +42h 0.96s
['z'] 0.05_10ENS 8번째 반복 +48h 0.96s
['z'] 0.05_10ENS 9번째 반복 +54h 0.96s
['z'] 0.05_10ENS 10번째 반복 +60h 0.96s
['z'] 0.05_10ENS 11번째 반복 +66h 0.96s
['z'] 0.05_10ENS 12번째 반복 +72h 0.96s
['z'] 0.05_10ENS 13번째 반복 +78h 0.96s
['z'] 0.05_10ENS 14번째 반복 +84h 0.96s
['z'] 0.05_10ENS 15번째 반복 +90h 0.96s
['z'] 0.05_10ENS 16번째 반복 +96h 0.96s
['z'] 0.05_10ENS 17번째 반복 +102h 0.96s
['z'] 0.05_10ENS 18번째 반복 +108h 0.96s
['z'] 0.05_10ENS 19번째 반복 +114h 0.96s
['z'] 0.05_10ENS 20번째 반복 +120h 0.96s
['z'] 0.05_10ENS 21번째 반복 +126h 0.98s
['z'] 0.05_10ENS 22번째 반복 +132h 0.99s
['z'] 0.05_10ENS 23번째 반복 +138h 0.99s
['z'] 0.05_10ENS 24번째 반복 +144h 0.98s
['z'] 0.05_10ENS 25번째 반복 +150h 0.97s
['z'] 0.05_10ENS 26번째 반복 +156h 0.96s
['z'] 0.05_10ENS 27번째 반복 +162h 0.98s
['z'] 0.05_10ENS 28번째 반복 +1